# 10 · Federated platinum score on NEPC tiers / AVPC within metastatic and mCRPC subgroups

The frozen platinum-trained federated bundles (refit without Dana-Farber: Fred Hutch, JHU and MSK) are scored on the
DFCI ADT cohort by `04_federated_transfer.ipynb`. This notebook reads those patient-level scores and asks how well the
score discriminates time to **NEPC** (tier sets t1, t1–t2, t1–t3), **AVPC** (strict) and **AVPC + adjacent**
(`st.ENDPOINTS`) within two subgroups:

1. **Metastatic** — `spec.metastatic_status`: `met_date` (mCRPC labels) on or before the landmark date is *yes*, dated
   later is *no*; undated is *no* if the AVPC label says non-metastatic and *date unknown* if it says metastatic.
2. **mCRPC** (nested in metastatic) — the mCRPC label as-is (`is_mcrpc`, `mcrpc_date`, including the drug-proxy route):
   `mcrpc_date` on or before the landmark date is *yes*; dated later or `is_mcrpc` False is *no*; `is_mcrpc` True without
   a date is *date unknown*.

The landmark date is the ADT anchor + landmark (0 / 90 / 180 days). Patients without a label row are *missing*.
Two families (XGBoost and the predictive checklist (raw item count, per model horizon)) and every landmark
are evaluated; config `both` only. Outcomes are each endpoint's incident landmark frame from 04, unchanged.

**Primary**: status at the landmark. **Sensitivity**: status *ever* (any metastatic / mCRPC label over the whole
observed course). That looks ahead, so it tests for signal and is not a deployable subgroup.

Each level reports n / events, Harrell's C, mean AUC(t) and HR per SD (on the full-frame score scale), with
patient-bootstrap 95% CIs (50 draws; `ext.subgroup_metrics`, event floor 10). No interaction tests. Only aggregate tables and PDFs are written.

In [ ]:
import os, sys
from pathlib import Path
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'survival_common').is_dir()), None)
if ROOT is None: raise RuntimeError('Run from the repository or COMPASS/notebooks')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'COMPASS' / 'federated_models'))
import numpy as np
import pandas as pd
from IPython.display import display
import reviewer_extended as ext
import reviewer_sources as src
import subgroup_transfer as st

DATA_ROOT = Path(os.getenv('COMPASS_DATA_ROOT', '/data/gusev/USERS/jpconnor/data/CAIA/COMPASS'))
RHINO_ROOT = Path(os.getenv('COMPASS_RHINO_ROOT', str(DATA_ROOT / 'rhino_data_v2')))
TRANSFER_DIR = Path(os.getenv('COMPASS_FEDERATED_TRANSFER_NO_DFCI', str(RHINO_ROOT / 'federated_transfer_adt_no_DFCI')))
SCORES_PATH = Path(os.getenv('COMPASS_REVIEWER_SCORES', str(TRANSFER_DIR / 'all_models' / 'transfer_scores.csv')))
LLM_ROOT = Path(os.getenv('LLM_ANNOTATIONS_DATA_PATH', '/data/gusev/USERS/jpconnor/data/LLM_annotations'))
LOCAL_PATH = Path(os.getenv('COMPASS_REVIEWER_LOCAL', str(DATA_ROOT / 'longitudinal_prediction_data_adt.csv')))
MCRPC_PATH = Path(os.getenv('COMPASS_REVIEWER_CRPC', str(LLM_ROOT / 'LLM_disease_course/mcrpc_labels.parquet')))
AVPC_PATH = Path(os.getenv('COMPASS_REVIEWER_AVPC', str(LLM_ROOT / 'LLM_avpc/avpc_labels.parquet')))
OUT = Path(os.getenv('COMPASS_FIG_ROOT', '/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS')) / 'subgroup_transfer'
CONFIG = os.getenv('COMPASS_REVIEWER_CONFIG', 'both')
N_BOOT = int(os.getenv('COMPASS_SUBGROUP_BOOTSTRAPS', '50'))
SEED = 0
print('Scores:', SCORES_PATH, '\nAggregate output:', OUT, '\nBootstrap draws:', N_BOOT)

## Load and check the transferred scores

Same checks as notebook 06: required columns, valid durations and events, and one row per patient per model identity.

In [ ]:
if not SCORES_PATH.exists():
    raise FileNotFoundError(f'{SCORES_PATH} missing. Run notebook 04 to export scores under the protected data root.')
scores = pd.read_csv(SCORES_PATH, dtype={'person_id': 'string'})
scores['person_id'] = pd.to_numeric(scores.person_id, errors='raise').astype('Int64').astype('string')
required = {'person_id', 'endpoint', 'model_family', 'bundle', 'landmark_days', 'config', 'risk_score', 'duration_days', 'event'}
missing = required - set(scores)
if missing: raise ValueError(f'Missing score columns: {sorted(missing)}')
for c in ['risk_score', 'duration_days', 'event', 'landmark_days']:
    scores[c] = pd.to_numeric(scores[c], errors='coerce')
scores = st.select_scores(scores, endpoints=st.ENDPOINTS, families=st.FAMILIES, config=CONFIG)
if scores[['risk_score', 'duration_days', 'event']].isna().any().any() or not scores.event.isin([0, 1]).all() or (scores.duration_days <= 0).any():
    raise ValueError('Invalid scores, durations, or events')
scores['landmark_days'] = scores.landmark_days.astype(int)
OUT.mkdir(parents=True, exist_ok=True)
display(scores.groupby(['endpoint', 'model_family', 'landmark_days', 'model_horizon_days'], dropna=False)
        .agg(n=('person_id', 'size'), events=('event', 'sum')).reset_index())

## Subgroup status

The anchor date comes from the ADT longitudinal file, `met_date`, `is_mcrpc` and `mcrpc_date` from the mCRPC labels,
and `is_metastatic` from the AVPC labels. Joins are one row per patient, and conflicting values stop execution.
The counts below are per endpoint × landmark frame. They are the same for every family scored on that frame, and
each variable's levels sum to the frame n.

In [ ]:
local = src.read(LOCAL_PATH)
meta = st.patient_frame(local, src.read(MCRPC_PATH), src.read(AVPC_PATH))
unscored = set(scores.person_id) - set(meta.person_id)
if unscored: raise ValueError(f'{len(unscored)} scored patients missing from the longitudinal file')
print(f'Patients: {len(meta):,}; mCRPC labelled: {meta.is_mcrpc.notna().sum():,}; '
      f'metastatic label: {meta.is_metastatic.notna().sum():,}')
subgroup_counts = st.counts(scores, meta)
subgroup_counts.to_csv(OUT / 'subgroup_counts.csv', index=False)
display(subgroup_counts.pivot_table(index=['variable', 'level'], columns=['endpoint', 'landmark_days'],
                                    values=['n', 'events'], aggfunc='first').fillna(0).astype(int))

## Primary: subgroup status at the landmark

One row per endpoint × family × landmark × model horizon × subgroup level. `full_cohort` is the whole landmark frame,
for reference. Levels with fewer than 10 events report counts only.

In [ ]:
primary = st.run(scores, meta, st.PRIMARY_VARIABLES, n_boot=N_BOOT, seed=SEED)
primary.to_csv(OUT / 'subgroup_metrics_at_landmark.csv', index=False)
display(primary[['endpoint', 'model_family', 'landmark_days', 'model_horizon_days', 'variable', 'level', 'n', 'events', 'c_index']])

## Sensitivity: subgroup status ever (retrospective)

In [ ]:
ever = st.run(scores, meta, st.EVER_VARIABLES, n_boot=N_BOOT, seed=SEED)
ever.to_csv(OUT / 'subgroup_metrics_ever.csv', index=False)
display(ever[['endpoint', 'model_family', 'landmark_days', 'model_horizon_days', 'variable', 'level', 'n', 'events', 'c_index']])

## Summary tables and forest plots

C-index (95% CI) with n (events), by subgroup level × landmark, one table per endpoint × family (× checklist horizon).
Forest plots show HR per SD of the full-frame score, one PDF per endpoint × family × horizon × timing.

In [ ]:
def _fmt(r):
    if pd.isna(r.get('c_index')): return f"n={r.n} ({r.events}) –"
    return f"{r.c_index:.3f} ({r.c_lower:.3f}–{r.c_upper:.3f}); n={r.n} ({r.events})"

def _horizon(h):
    return '' if pd.isna(h) else f'_h{int(h)}'

for timing, table in (('at_landmark', primary), ('ever', ever)):
    t = table.assign(cell=table.apply(_fmt, axis=1))
    for (ep, fam, h), g in t.groupby(['endpoint', 'model_family', 'model_horizon_days'], dropna=False):
        print(f'\n{ep.upper()} · {fam}{_horizon(h)} · {timing}')
        display(g.pivot_table(index=['variable', 'level'], columns='landmark_days', values='cell', aggfunc='first'))
        f = g.assign(label='lm' + g.landmark_days.astype(str) + ' · ' + g.variable + ' · ' + g.level)
        ext.plot_forest(f, OUT / f'forest_{ep}_{fam}{_horizon(h)}_{timing}.pdf',
                        f'{ep.upper()} · {fam}{_horizon(h)} · subgroups {timing.replace("_", " ")}')
print('wrote aggregate outputs ->', OUT)

## Interpretation notes

- AVPC requires metastatic disease plus castration resistance (or C1), so most AVPC events can only occur in patients
  who become mCRPC. Within *no* levels, AVPC events are structurally rare and estimates may be suppressed. AVPC + adjacent also counts
  adjacent criteria met without the mCRPC gate, so it is less concentrated in mCRPC.
- mCRPC at landmark 0 is uncommon in an ADT-start cohort, so that cell will be small. Read n and events with every C-index.
- The *ever* analysis conditions on the future course and is a signal test, not a deployable subgroup.
- HR per SD uses the full-frame score scale within each model, so levels of the same model are comparable. Raw
  checklist item counts are not comparable to the continuous scores of the other families.
- Cox and KM quantities treat competing death as censoring. Bootstrap intervals condition on the frozen scores.
- The mCRPC label includes the drug-proxy route (a CRPC-only agent start), so a proxy-dated mCRPC is counted as
  mCRPC from that date.